# 第 5 章 · Tool Use（Google Search，ADK）

`google_search` 是**内建工具**（built-in tool），不是你写的 `FunctionTool`：
搜索在 Google 服务端执行，代码里看不到「调用 → 返回」这一跳，模型只拿到结果摘要。

这就是本章最后一个要点——**工具的三种来源**：自己写的函数（`FunctionTool`）、
框架内建的（`google_search`）、以及下面 `Executing_Code` 那种"模型自己写代码并执行"。

kernel 选 `Python (agentic-adk)`。书里原版跑不起来，三处【适配】已标。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】向上找含 common/ 的目录；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, describe_adk, short_error, retry_call
from google.adk.agents import LlmAgent   # 【适配】书里 from ... import Agent 却用 ADKAgent，会 NameError
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

# ★ 本章新增 google_search：内建工具。它是"服务端工具"，不是本地函数 ——
#   所以 tools=[google_search] 里放的不是你写的函数，而是框架导出的一个工具对象。
USER_ID = "user1234"

root_agent = LlmAgent(
    name="basic_search_agent",
    model=adk_model(),                   # 【适配】书里 gemini-2.0-flash-exp 已下线
    description="Agent to answer questions using Google Search.",
    instruction="I can answer your questions by searching the internet. Just ask me anything!",
    tools=[google_search],
)


def _text(content) -> str:
    """【适配】Content 没有 .text 属性，得走 parts。"""
    if getattr(content, "text", None):
        return content.text
    return "".join(p.text for p in (getattr(content, "parts", None) or [])
                   if getattr(p, "text", None))


runner = InMemoryRunner(root_agent)
session_id = str(uuid.uuid4())
# 【适配】create_session 在 ADK 2.13 是协程，同步上下文用 create_session_sync
runner.session_service.create_session_sync(
    app_name=runner.app_name, user_id=USER_ID, session_id=session_id
)

# 【适配】书里 nest_asyncio + asyncio.run(...)；notebook 的 kernel 本身已在事件循环里，
#   那句会报 "cannot be called from a running event loop"。runner.run 是同步生成器，直接用即可。
# 【适配】中转站偶尔会 ReadTimeout（不是我们的 bug，网关抖动）。把整段事件流收成 list，
#   再交给 retry_call 重试 —— 超时属于瞬时错误，重跑一次通常就过。
def collect_events():
    return list(runner.run(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text="what's the latest ai news?")]),
    ))


try:
    for event in retry_call(collect_events, label="搜索 "):
        if event.is_final_response() and event.content:
            print("Agent Response:", _text(event.content).strip())

            # 顺手验一下"真搜了没有"：有 grounding 就说明搜索确实发生了，
            # 没有则可能只是模型在凭记忆回答（内建工具的一个坑：调没调它不告诉你）。
            gm = getattr(event, "grounding_metadata", None)
            queries = list(getattr(gm, "web_search_queries", None) or []) if gm else []
            chunks = list(getattr(gm, "grounding_chunks", None) or []) if gm else []
            print(f"\n[grounding] 搜索词: {queries or '(无)'}")
            print(f"[grounding] 引用来源: {len(chunks)} 个")
except Exception as e:
    print(f"❌ {type(e).__name__}: {short_error(e)}")

Agent Response: The landscape of artificial intelligence is moving incredibly fast. As of **late October and early November 2026**, here is a breakdown of the most significant breakthroughs, corporate updates, and regulatory battles shaping the AI industry:

### 1. Major Model & Platform Releases
*   **OpenAI Rolls Out GPT-6 & GPT-6.1 Terra:** OpenAI has officially introduced GPT-6 to its platform (including the free tier). Alongside it, they launched **GPT-6.1 Terra**, a highly efficient mid-tier model designed to read screens, perform actions across apps, and output natural language at a fraction of the cost of its larger siblings, Sol and Astra.
*   **Mistral AI’s "Le Chonk":** European AI champion Mistral released **Mistral Large 4**, nicknamed "Le Chonk". This multimodal open-weight model crosses the threshold of **1 trillion parameters**, using a Mixture-of-Experts (MoE) architecture that only activates 49 billion parameters per token, making it incredibly powerful yet efficient.